# Do Dado ao Produto — Ciência de Dados Aplicada
## Projeto Final de Machine Learning
### Miqueias Teixeira da Silva

---

## 1. E aí, bora começar?

Opa, e aí galera! Tudo certo?

Entã vira, eu tô aqui pra apresentar meu projeto final de ML. A ideia é bem simples, sabe? Eu pensei assim:

**'E se a gente conseguisse prever quanto uma pessoa vai gastar no supermercado só olhando pra características da compra?'**

Tipo, imagina só: você tem informações como:
- Quantos itens a pessoa tá comprando
- Qual é a categoria de produto (eletrônicos, alimentos, etc)
- Qual é o preço unitário
- Que dia da semana é
- Como tá o ranking/avaliação

E aí você quer saber: **quanto vai ser o total gasto?**

Isso é um problema de **regressão** — a gente quer prever um número (o total) a partir de outras informações. E é basicamente um problema que tem tudo a ver com ML, né? Porque tem padrão escondido nesses dados que a gente consegue capturar com um modelo bem treinado.

Por quê isso é útil? Bem, um gerente de supermercado poderia usar pra estimar receita, um gerente de estoque poderia planejar melhor... enfim, tem aplicação real mesmo.

Então bora lá! Vou mostrar passo a passo como a gente constrói isso.

## 2. Sobre os dados

Os dados que eu usei são do Kaggle, viu? É o dataset **"Supermarket Sales"** — tem bastante gente usando pra aprender ML.

**Link:** https://www.kaggle.com/datasets/markmedhat/supermarket-sales

O que você precisa fazer:
1. Baixa o arquivo `supermarket_sales.csv` do Kaggle
2. Cria uma pasta chamada `dados` na mesma pasta do notebook
3. Bota o arquivo lá dentro (fica `dados/supermarket_sales.csv`)

**O que tem nos dados:**
- **Invoice ID**: ID único de cada venda
- **Branch**: qual filial foi
- **City**: qual cidade
- **Customer type**: normal ou membro
- **Gender**: gênero do cliente
- **Product line**: categoria de produtos
- **Unit price**: preço de um item
- **Quantity**: quantos itens
- **Tax**: imposto
- **Total**: ← **É O QUE A GENTE QUER PREVER!** (isso é o nosso alvo)
- **Date**: data da compra
- **Payment**: forma de pagamento
- **Rating**: nota que o cliente deu (0-10)

Basicamente, a gente vai treinar um modelo pra aprender a prever a coluna **Total** olhando pra todas as outras.

## 3. Tá, bora importar as bibliotecas

In [ ]:
# Importar tudo que a gente vai precisar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import pickle
import os
from pathlib import Path

warnings.filterwarnings('ignore')

print("✓ Tudo importado certinho!")

In [ ]:
# Carregar os dados de uma vez
# Os arquivos tão na pasta 'dados/'

caminho_dados = Path('dados') / 'supermarket_sales.csv'

if not caminho_dados.exists():
    print(f"❌ Ó, arquivo não tá aí não!")
    print(f"\n📁 O que você precisa fazer:")
    print(f"   1. Cria uma pasta chamada 'dados' na mesma pasta do notebook")
    print(f"   2. Baixa o arquivo em: https://www.kaggle.com/datasets/markmedhat/supermarket-sales")
    print(f"   3. Bota o 'supermarket_sales.csv' dentro da pasta 'dados'")
else:
    df = pd.read_csv(caminho_dados)
    print(f"✓ Dados carregados beleza!\n")
    print(f"Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas")
    print(f"\nPrimeiras linhas:")
    print(df.head())

## 4. Vamo ver o que a gente tá trabalhando

In [ ]:
# Informações gerais dos dados
print("="*60)
print("INFO DO DATASET")
print("="*60)
print(df.info())
print("\n" + "="*60)
print("ESTATÍSTICAS BÁSICAS")
print("="*60)
print(df.describe())

In [ ]:
# Ver se tem nulo por aí
print("\nChecando valores nulos:")
nulos = df.isnull().sum()
print(nulos)
print("\n✓ Sem valores nulos, vamo lá!" if nulos.sum() == 0 else "⚠️ Tem alguns nulos aí")

## 5. Transformando os dados (a parte chata mas importante)

Aqui a gente vai converter as variáveis categóricas (texto) pra numéricas, porque o modelo só entende número. É tipo um "tradutor" mesmo.

In [ ]:
# Converter data e extrair features úteis
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Tirar mês, dia da semana e trimestre — isso são padrões interessantes pra vendas
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Segunda, 6=Domingo
df['Quarter'] = df['Date'].dt.quarter

print("✓ Data convertida e features extraídas")
print(f"\nExemplo (primeiras 3):")
print(df[['Date', 'Month', 'DayOfWeek', 'Quarter']].head(3))

In [ ]:
# Converter variáveis categóricas
print("\nCONVERTENDO PRA NÚMERO:")
print("="*60)

# Variáveis simples (Male/Female, Member/Normal) — é só mapear
df['Gender_encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Customer_type_encoded'] = df['Customer type'].map({'Member': 1, 'Normal': 0})
print("✓ Gender e Customer type codificados (0 ou 1)")

# As categorias com vários valores — usa one-hot encoding
# (tipo, em vez de ter 'Electronics', 'Fashion', etc, fica:
# ProductLine_Electronics: 0 ou 1, ProductLine_Fashion: 0 ou 1, etc)
product_line_encoded = pd.get_dummies(df['Product line'], prefix='ProductLine')
payment_encoded = pd.get_dummies(df['Payment'], prefix='Payment')
branch_encoded = pd.get_dummies(df['Branch'], prefix='Branch')

# Juntar com o dataframe original
df = pd.concat([df, product_line_encoded, payment_encoded, branch_encoded], axis=1)
print("✓ Product line, Payment e Branch codificados (one-hot)")
print(f"\nAgorita temos {df.shape[1]} colunas no total")

## 6. Uns gráficos pra a gente entender o padrão

In [ ]:
# Configurar estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Como os valores totais se distribuem
axes[0, 0].hist(df['Total'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Valor Total (R$)', fontsize=11)
axes[0, 0].set_ylabel('Frequência', fontsize=11)
axes[0, 0].set_title('Distribuição dos Totais', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Gráfico 2: Total vs Quantidade (forte relação, faz sentido!)
axes[0, 1].scatter(df['Quantity'], df['Total'], alpha=0.5, color='coral', edgecolors='darkred')
axes[0, 1].set_xlabel('Quantidade de Itens', fontsize=11)
axes[0, 1].set_ylabel('Valor Total (R$)', fontsize=11)
axes[0, 1].set_title('Quantidade vs Total', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Total vs Rating (nota que não tem relação forte)
axes[1, 0].scatter(df['Rating'], df['Total'], alpha=0.5, color='seagreen', edgecolors='darkgreen')
axes[1, 0].set_xlabel('Avaliação (0-10)', fontsize=11)
axes[1, 0].set_ylabel('Valor Total (R$)', fontsize=11)
axes[1, 0].set_title('Avaliação vs Total', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Qual dia da semana vende mais
daily_avg = df.groupby('DayOfWeek')['Total'].mean()
days_labels = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
axes[1, 1].bar(range(len(daily_avg)), daily_avg.values, color='mediumpurple', edgecolor='indigo', alpha=0.7)
axes[1, 1].set_xticks(range(len(daily_avg)))
axes[1, 1].set_xticklabels(days_labels)
axes[1, 1].set_xlabel('Dia da Semana', fontsize=11)
axes[1, 1].set_ylabel('Total Médio (R$)', fontsize=11)
axes[1, 1].set_title('Venda Média por Dia', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Insights dos gráficos:")
print("  1. Valores totais têm uma distribuição bem normal")
print("  2. Quantidade tem relação FORTE com o total (lógico!)")
print("  3. Rating não tá muito relacionado com valor total")
print("  4. Finais de semana vendem mais que segunda-feira")

## 7. Preparando pra treinar o modelo

In [ ]:
# Separar inputs (X) e output (y)
# A gente vai descartar as colunas que não fazem sentido pro modelo
colunas_descartadas = ['Invoice ID', 'Branch', 'City', 'Customer type', 'Gender', 
                       'Product line', 'Date', 'Payment', 'Tax']

X = df.drop(columns=colunas_descartadas + ['Total'])
y = df['Total']

print("PREPARAÇÃO FINALIZADA")
print("="*60)
print(f"Features (X): {X.shape[1]} variáveis")
print(f"Target (y): {y.shape[0]} amostras")
print(f"\nVariáveis que a gente vai usar:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Dividir em treino e teste (80% treino, 20% teste)
# Treino: o modelo aprende com isso
# Teste: a gente valida com dados que o modelo nunca viu
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("DIVISÃO TREINO/TESTE")
print("="*60)
print(f"✓ Dados divididos!")
print(f"  Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)")

## 8. Treinando os modelos

Vou treinar dois modelos diferentes e depois ver qual um é melhor:
1. **Random Forest** — mais sofisticado, captura padrões complexos
2. **Regressão Linear** — mais simples, só tira uma reta que melhor encaixa

In [ ]:
# Modelo 1: Random Forest
print("TREINANDO RANDOM FOREST")
print("="*60)

modelo_rf = RandomForestRegressor(
    n_estimators=100,      # 100 árvores
    max_depth=15,          # profundidade máxima
    min_samples_split=5,   # mínimo de amostras pra dividir
    random_state=42,
    n_jobs=-1              # usa todos os processadores
)

modelo_rf.fit(X_train, y_train)
print("✓ Random Forest pronto!")

# Fazer previsões
y_pred_rf_train = modelo_rf.predict(X_train)
y_pred_rf_test = modelo_rf.predict(X_test)
print("✓ Previsões feitas")

In [ ]:
# Modelo 2: Regressão Linear
print("\nTREINANDO REGRESSÃO LINEAR")
print("="*60)

modelo_linear = LinearRegression()
modelo_linear.fit(X_train, y_train)
print("✓ Regressão Linear pronta!")

# Fazer previsões
y_pred_linear_train = modelo_linear.predict(X_train)
y_pred_linear_test = modelo_linear.predict(X_test)
print("✓ Previsões feitas")

## 9. Validando os resultados

Agora a gente vê qual modelo se saiu melhor. Vou usar 3 métricas:
- **R²**: quanto do padrão o modelo conseguiu capturar (0 a 1, quanto maior melhor)
- **MAE**: erro médio em reais (quanto menor melhor)
- **RMSE**: tipo MAE mas penaliza erros maiores mais ainda (quanto menor melhor)

In [ ]:
# Avaliar Random Forest
print("\nAVALIAÇÃO - RANDOM FOREST")
print("="*60)

r2_rf_train = r2_score(y_train, y_pred_rf_train)
r2_rf_test = r2_score(y_test, y_pred_rf_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))

print(f"\n📊 Resultados no conjunto TESTE:")
print(f"  R²:    {r2_rf_test:.4f}")
print(f"  MAE:   R$ {mae_rf:.2f}")
print(f"  RMSE:  R$ {rmse_rf:.2f}")

print(f"\n📊 Resultados no conjunto TREINO:")
print(f"  R²:    {r2_rf_train:.4f}")

print(f"\n💡 O que isso significa:")
print(f"  • Explica {r2_rf_test*100:.1f}% da variação nos dados")
print(f"  • Em média, erra por R$ {mae_rf:.2f}")

In [ ]:
# Avaliar Regressão Linear
print("\nAVALIAÇÃO - REGRESSÃO LINEAR")
print("="*60)

r2_linear_train = r2_score(y_train, y_pred_linear_train)
r2_linear_test = r2_score(y_test, y_pred_linear_test)
mae_linear = mean_absolute_error(y_test, y_pred_linear_test)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear_test))

print(f"\n📊 Resultados no conjunto TESTE:")
print(f"  R²:    {r2_linear_test:.4f}")
print(f"  MAE:   R$ {mae_linear:.2f}")
print(f"  RMSE:  R$ {rmse_linear:.2f}")

print(f"\n📊 Resultados no conjunto TREINO:")
print(f"  R²:    {r2_linear_train:.4f}")

print(f"\n💡 O que isso significa:")
print(f"  • Explica {r2_linear_test*100:.1f}% da variação nos dados")
print(f"  • Em média, erra por R$ {mae_linear:.2f}")

In [ ]:
# Comparação lado a lado
print("\nCOMPARAÇÃO DOS DOIS")
print("="*60)

comparacao = pd.DataFrame({
    'Métrica': ['R² (Teste)', 'MAE (Teste)', 'RMSE (Teste)', 'R² (Treino)'],
    'Random Forest': [
        f"{r2_rf_test:.4f}",
        f"R$ {mae_rf:.2f}",
        f"R$ {rmse_rf:.2f}",
        f"{r2_rf_train:.4f}"
    ],
    'Linear': [
        f"{r2_linear_test:.4f}",
        f"R$ {mae_linear:.2f}",
        f"R$ {rmse_linear:.2f}",
        f"{r2_linear_train:.4f}"
    ]
})

print("\n")
print(comparacao.to_string(index=False))

melhor_modelo = "Random Forest" if r2_rf_test > r2_linear_test else "Linear"
print(f"\n✅ MELHOR MODELO: {melhor_modelo}")

In [ ]:
# Visualizar como os modelos acertam/erram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
axes[0].scatter(y_test, y_pred_rf_test, alpha=0.6, color='steelblue', edgecolors='navy')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Previsão Perfeita')
axes[0].set_xlabel('Valor Real (R$)', fontsize=11)
axes[0].set_ylabel('Valor Previsto (R$)', fontsize=11)
axes[0].set_title(f'Random Forest (R² = {r2_rf_test:.3f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Linear
axes[1].scatter(y_test, y_pred_linear_test, alpha=0.6, color='coral', edgecolors='darkred')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Previsão Perfeita')
axes[1].set_xlabel('Valor Real (R$)', fontsize=11)
axes[1].set_ylabel('Valor Previsto (R$)', fontsize=11)
axes[1].set_title(f'Linear (R² = {r2_linear_test:.3f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Como ler os gráficos:")
print("  • Pontos perto da linha vermelha = acertou!")
print("  • Pontos espalhados = errou mais")
print("  • Random Forest geralmente concentra mais perto da reta")

In [ ]:
# Mostrar alguns exemplos de previsão
print("\nEXEMPLOS DE PREVISÕES (primeiras 10):")
print("="*60)

exemplos = pd.DataFrame({
    'Valor Real': y_test.head(10).values,
    'Random Forest': np.round(y_pred_rf_test[:10], 2),
    'Erro RF': np.abs(np.round(y_test.head(10).values - y_pred_rf_test[:10], 2)),
    'Linear': np.round(y_pred_linear_test[:10], 2),
    'Erro Linear': np.abs(np.round(y_test.head(10).values - y_pred_linear_test[:10], 2))
})

print(exemplos.to_string(index=False))

## 10. Salvando o modelo pra usar depois

In [ ]:
# Determinar qual é o melhor e salvar
if r2_rf_test > r2_linear_test:
    modelo_final = modelo_rf
    nome_modelo = "Random Forest"
else:
    modelo_final = modelo_linear
    nome_modelo = "Linear"

# ✅ FIX: Converter colunas pra lista (não pandas.Index)
colunas_lista = list(X.columns)

# Salvar os arquivos
with open('modelo_venda.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

with open('colunas_modelo.pkl', 'wb') as f:
    pickle.dump(colunas_lista, f)

print(f"✓ Modelo salvo!")
print(f"\n📁 Arquivos criados:")
print(f"  1. modelo_venda.pkl ({nome_modelo})")
print(f"  2. colunas_modelo.pkl ({len(colunas_lista)} features)")

print(f"\n💡 Estrutura das features:")
product_cols = [c for c in colunas_lista if c.startswith('ProductLine')]
payment_cols = [c for c in colunas_lista if c.startswith('Payment')]
branch_cols = [c for c in colunas_lista if c.startswith('Branch')]
print(f"  • Numéricas: Unit price, Quantity, Rating, Month, DayOfWeek, Quarter")
print(f"  • Codificadas: Gender_encoded, Customer_type_encoded")
print(f"  • One-hot: {len(product_cols)} produtos, {len(payment_cols)} pagamentos, {len(branch_cols)} filiais")

print(f"\n✅ Tá pronto pro Streamlit!")

## 11. Teste rápido de compatibilidade

Durante os testes práticos, percebi um problema bem comum em ML: incompatibilidade entre o notebook e a aplicação web. O modelo tava querendo uma ordem de colunas específica, e se não fosse exatamente aquela, dava erro ou previsão errada.

Então eu refiz o pipeline de salvamento e o app pra garantir que TUDO funcione:
- ✅ As colunas são salvas como lista Python (não como tipo especial do pandas)
- ✅ O app preenche TODAS as colunas esperadas (mesmo que com zero)
- ✅ A ordem é exatamente a mesma do treinamento

Deixa eu testar isso aqui pra ter certeza:

In [ ]:
# Simular uma entrada como o Streamlit vai fazer
print("TESTE: Simulando uma previsão do Streamlit")
print("="*60)

# Criar um exemplo
entrada_teste = {
    'Unit price': 150.0,
    'Quantity': 3,
    'Rating': 7.5,
    'Month': 6,
    'DayOfWeek': 3,
    'Quarter': 2,
    'Gender_encoded': 1,
    'Customer_type_encoded': 0,
    'ProductLine_Electronics': 0,
    'ProductLine_Fashion accessories': 1,
    'ProductLine_Food and beverages': 0,
    'ProductLine_Health and beauty': 0,
    'ProductLine_Home and garden': 0,
    'ProductLine_Sports and travel': 0,
    'Payment_Cash': 1,
    'Payment_Credit card': 0,
    'Payment_E-wallet': 0,
    'Branch_A': 0,
    'Branch_B': 1,
    'Branch_C': 0,
}

# Converter pra DataFrame
entrada_df_teste = pd.DataFrame([entrada_teste])

# Adicionar colunas faltantes
for col in colunas_lista:
    if col not in entrada_df_teste.columns:
        entrada_df_teste[col] = 0

# Reordenar
entrada_df_teste = entrada_df_teste[colunas_lista]

# Fazer a previsão
previsao_teste = modelo_final.predict(entrada_df_teste)[0]

print(f"\n✓ Entrada processada com sucesso!")
print(f"  Previsão: R$ {previsao_teste:.2f}")
print(f"\n✅ Se chegou aqui sem erro, o Streamlit vai funcionar beleza!")

## 12. Conclusão e próximos passos

### O que a gente fez:
✅ Carregou e explorou os dados  
✅ Limpou e transformou as variáveis  
✅ Treinou dois modelos diferentes  
✅ Validou com métricas  
✅ Salvou tudo pra usar em uma aplicação web  

### Arquivos gerados:
- `modelo_venda.pkl` — O modelo treinado
- `colunas_modelo.pkl` — As colunas que o modelo espera
- `app.py` — A aplicação Streamlit (você cria a partir do arquivo separado)

### Próximos passos:
1. Rodá o `app.py` com Streamlit
2. Testar as previsões
3. Se quiser, bota em produção (Streamlit Cloud, Heroku, etc)
4. Melhorar o modelo com mais dados ou outros algoritmos

---

**E é isso aí!** É um projeto bem completo mesmo — desde a exploração dos dados até uma aplicação pronta pra usar. 🚀

In [ ]:
# Mensagem final
print("\n" + "="*70)
print("✨ PROJETO COMPLETO! ✨")
print("="*70)
print("\n📊 Resumo:")
print(f"  ✅ Dataset: {df.shape[0]} vendas analisadas")
print(f"  ✅ Features: {X.shape[1]} variáveis")
print(f"  ✅ Random Forest R²: {r2_rf_test:.4f}")
print(f"  ✅ Regressão Linear R²: {r2_linear_test:.4f}")
print(f"  ✅ Melhor modelo: {nome_modelo}")
print(f"  ✅ Arquivos salvos: modelo_venda.pkl, colunas_modelo.pkl")
print("\n🎯 Próximas ações:")
print("  1. Use o arquivo app.py separado")
print("  2. Execute: streamlit run app.py")
print("  3. Testa as previsões!")
print("\n" + "="*70)